# 02. Sensitivity Analysis

This notebook runs the sensitivity analysis of trained RL policies under different supply chain scenarios. It analyzes policy performance with respect to changes in hospital demand level variances, shelf lives, disruption frequencies, and costs.

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

from src.configs.config import MODELS_DIR, TABLES_DIR
from src.env.platelet_env import PlateletSupplyChainEnv
from src.agents.mappo import MultiAgentMAPPO
from src.agents.ppo import SingleAgentPPO
from src.evaluation.evaluation import evaluate_multi_seed

### Helper function to run sensitivity analysis for an architecture
Evaluates trained policies against modified environment config parameters.

In [ ]:
def run_sensitivity(agent_class, is_mappo, action_dim, name, model_prefix, env_with_trans, excel_filename):
    scenarios = {
        'Base': {},
        'ShelfLife_3': {'shelf_life': 3},
        'ShelfLife_4': {'shelf_life': 4},
        'Wastage_Cost_5': {'wastage_cost': 5.0},
        'Wastage_Cost_10': {'wastage_cost': 10.0},
        'Shortage_Cost_15': {'shortage_cost': 15.0},
        'Shortage_Cost_20': {'shortage_cost': 20.0},
        'Disruption_Prob_0.01': {'disruption_prob': 0.01},
        'Disruption_Prob_0.05': {'disruption_prob': 0.05},
        'Disruption_Mult_1.2': {'disruption_mult': 1.2},
        'Disruption_Mult_1.7': {'disruption_mult': 1.7},
        'Disruption_Mult_2.0': {'disruption_mult': 2.0},
    }

    seeds = [42, 100, 2023, 888, 55]
    all_rows = []

    for scen_name, overrides in scenarios.items():
        print(f"Processing scenario: {scen_name} for {name}")
        temp_env = PlateletSupplyChainEnv(env_config=overrides, enable_transshipment=env_with_trans)
        
        for s in seeds:
            model_path = os.path.join(MODELS_DIR, f"{model_prefix}_seed_{s}.pt")
            if not os.path.exists(model_path):
                print(f"⚠️ Model file missing: {model_path}")
                continue
            
            # Load policy
            if is_mappo:
                agent = MultiAgentMAPPO()
            else:
                agent = SingleAgentPPO(action_dim=action_dim)
            agent.load(model_path)
            
            # Evaluate
            h1_sum, h2_sum, sc_sum, _ = evaluate_multi_seed(temp_env, agent, train_seed=s, is_mappo=is_mappo, eval_seeds=range(1, 51))
            
            all_rows.append({
                'Scenario': scen_name,
                'Train_Seed': s,
                'SC_Network_Total_Cost_Mean': sc_sum['SC_Network_Total_Cost_Mean_Mean'].values[0],
                'SC_Network_Total_Cost_Std': sc_sum['SC_Network_Total_Cost_Mean_Std'].values[0],
                'Shortage_Rate_Pct_Mean': sc_sum['Shortage_Rate_Pct_Mean_Mean'].values[0],
                'Wastage_Rate_Pct_Mean': sc_sum['Wastage_Rate_Pct_Mean_Mean'].values[0],
                'Transshipment_Cost_Mean': sc_sum['Transshipment_Cost_Mean_Mean'].values[0],
            })
            
    df_res = pd.DataFrame(all_rows)
    os.makedirs(TABLES_DIR, exist_ok=True)
    df_res.to_excel(os.path.join(TABLES_DIR, excel_filename), index=False)
    print(f"✅ Saved sensitivity results to {excel_filename}")

In [ ]:
# Run Sensitivity Analyses
run_sensitivity(MultiAgentMAPPO, True, 3, "MAPPO", "mappo", True, "Sensitivity_MAPPO.xlsx")
run_sensitivity(SingleAgentPPO, False, 3, "SA-PPO (with Trans)", "sa_t", True, "Sensitivity_SA_T.xlsx")
run_sensitivity(SingleAgentPPO, False, 2, "SA-PPO (No Trans)", "sa_nt", False, "Sensitivity_SA_NT.xlsx")